In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
!pip install requests pandas pyarrow -q
# import doi_enrichment

In [ ]:
#!/usr/bin/env python3
"""
doi_enrichment.py  —  Google Colab
==================================
Enrich on oa_doi from two open sources:

  UNPAYWALL      -> authoritative OA status + OA color (gold/green/hybrid/bronze/
                    closed) + best OA location. This is the REFERENCE STANDARD for
                    OA; oa_is_oa / oa_oa_status are OpenAlex's guess. OA is a
                    required CONFOUNDER in Paper 3's citation-penalty matching, so
                    you want the authoritative version and a flag where the two
                    sources disagree.
  OPENCITATIONS  -> an INDEPENDENT citation count (COCI/OpenCitations Index), a
                    cross-check on OpenAlex cited_by_count for the robustness note.

Same resumable pattern as the funding harvest: threaded per-DOI, cached to JSONL
keyed on oa_id, rerun to continue after a disconnect.

INPUT: your latest merged parquet (must have oa_doi; oa_is_oa/oa_oa_status
       optional, used only for the agreement report).

SETUP
  !pip install requests pandas pyarrow -q
  # Unpaywall REQUIRES an email in the query (it's free, no key). Set MAILTO.
  # OpenCitations works keyless; an access token raises your rate limit if you
  # have one (https://opencitations.net) — optional.
"""

import json, re, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd, requests

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
IN_PARQUET  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/llm_screen/screened_llm.parquet"
OUT_PARQUET = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_upw_oc.parquet"
CACHE_UPW   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/upw_oc_enrich/unpaywall.jsonl"
CACHE_OC    = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/upw_oc_enrich/opencitations.jsonl"
MAILTO      = "cognosxx@gmail.com"     # REQUIRED by Unpaywall
OC_TOKEN    = ""                          # optional OpenCitations access token
UPW_WORKERS = 8
OC_WORKERS  = 6

Path(CACHE_UPW).parent.mkdir(parents=True, exist_ok=True)
Path(CACHE_OC).parent.mkdir(parents=True, exist_ok=True)


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)

def norm_doi(x):
    d = S(x).lower().replace("https://doi.org/", "").replace("doi.org/", "").strip()
    return d or ""

def load_cache(path):
    done = {}
    if Path(path).exists():
        for line in open(path):
            try:
                d = json.loads(line); done[d["oa_id"]] = d
            except Exception:
                pass
    print(f"[cache] {len(done):,} in {Path(path).name}", file=sys.stderr)
    return done


# =========================================================================== #
# 1. UNPAYWALL  <- oa_doi                                                       #
# =========================================================================== #
def _upw_one(oa_id, doi, sess):
    for attempt in range(4):
        try:
            r = sess.get(f"https://api.unpaywall.org/v2/{doi}",
                         params={"email": MAILTO}, timeout=30)
            if r.status_code == 200:
                j = r.json()
                loc = j.get("best_oa_location") or {}
                return {"oa_id": oa_id,
                        "upw_is_oa": bool(j.get("is_oa")),
                        "upw_oa_status": j.get("oa_status", ""),          # gold/green/hybrid/bronze/closed
                        "upw_is_gold_doaj": bool(j.get("journal_is_in_doaj")),
                        "upw_has_repo_copy": bool(j.get("has_repository_copy")),
                        "upw_host_type": loc.get("host_type", "") if loc else "",
                        "upw_oa_url": loc.get("url", "") if loc else ""}
            if r.status_code == 404:
                return {"oa_id": oa_id, "upw_is_oa": None, "upw_oa_status": "not_found"}
            if r.status_code in (422,):     # malformed DOI
                return {"oa_id": oa_id, "upw_is_oa": None, "upw_oa_status": "invalid_doi"}
            if r.status_code in (429, 500, 502, 503):
                time.sleep(2 ** attempt); continue
            return {"oa_id": oa_id, "upw_is_oa": None, "upw_oa_status": str(r.status_code)}
        except requests.RequestException:
            time.sleep(2 ** attempt)
    return {"oa_id": oa_id, "upw_is_oa": None, "upw_oa_status": "failed"}

def harvest_unpaywall(df):
    cache = load_cache(CACHE_UPW)
    sub = df[df["oa_doi"].map(lambda x: bool(norm_doi(x)))]
    todo = [(r["oa_id"], norm_doi(r["oa_doi"])) for _, r in sub.iterrows()
            if r["oa_id"] not in cache]
    print(f"[unpaywall] {len(todo):,} DOIs to query", file=sys.stderr)
    fh = open(CACHE_UPW, "a"); sess = requests.Session()
    sess.headers.update({"User-Agent": f"tick-sos-study/1.0 (mailto:{MAILTO})"})
    with ThreadPoolExecutor(max_workers=UPW_WORKERS) as ex:
        futs = [ex.submit(_upw_one, oid, doi, sess) for oid, doi in todo]
        for n, fut in enumerate(as_completed(futs), 1):
            fh.write(json.dumps(fut.result()) + "\n"); fh.flush()
            if n % 500 == 0:
                print(f"[unpaywall] {n:,}/{len(todo):,}", file=sys.stderr)
    fh.close()
    return load_cache(CACHE_UPW)


# =========================================================================== #
# 2. OPENCITATIONS  <- oa_doi  (independent citation count)                    #
# =========================================================================== #
def _oc_one(oa_id, doi, sess):
    headers = {"authorization": OC_TOKEN} if OC_TOKEN else {}
    for attempt in range(4):
        try:
            r = sess.get(f"https://opencitations.net/index/api/v2/citation-count/doi:{doi}",
                         headers=headers, timeout=30)
            if r.status_code == 200:
                j = r.json()
                cnt = int(j[0]["count"]) if j else 0
                return {"oa_id": oa_id, "oc_cited_by": cnt}
            if r.status_code in (429, 500, 502, 503):
                time.sleep(2 ** attempt); continue
            return {"oa_id": oa_id, "oc_cited_by": None}
        except (requests.RequestException, ValueError, KeyError, IndexError):
            time.sleep(2 ** attempt)
    return {"oa_id": oa_id, "oc_cited_by": None}

def harvest_opencitations(df):
    cache = load_cache(CACHE_OC)
    sub = df[df["oa_doi"].map(lambda x: bool(norm_doi(x)))]
    todo = [(r["oa_id"], norm_doi(r["oa_doi"])) for _, r in sub.iterrows()
            if r["oa_id"] not in cache]
    print(f"[opencitations] {len(todo):,} DOIs to query", file=sys.stderr)
    fh = open(CACHE_OC, "a"); sess = requests.Session()
    with ThreadPoolExecutor(max_workers=OC_WORKERS) as ex:
        futs = [ex.submit(_oc_one, oid, doi, sess) for oid, doi in todo]
        for n, fut in enumerate(as_completed(futs), 1):
            fh.write(json.dumps(fut.result()) + "\n"); fh.flush()
            if n % 500 == 0:
                print(f"[opencitations] {n:,}/{len(todo):,}", file=sys.stderr)
    fh.close()
    return load_cache(CACHE_OC)


# =========================================================================== #
# 3. RECONCILE + write + agreement report                                     #
# =========================================================================== #
def reconcile(df, upw, oc):
    upw_df = pd.DataFrame(list(upw.values())) if upw else pd.DataFrame(columns=["oa_id"])
    oc_df  = pd.DataFrame(list(oc.values()))  if oc  else pd.DataFrame(columns=["oa_id"])
    out = df.merge(upw_df, on="oa_id", how="left").merge(oc_df, on="oa_id", how="left")

    # OA confounder: prefer Unpaywall; flag where it disagrees with OpenAlex
    if "oa_is_oa" in out.columns:
        oa_guess = out["oa_is_oa"].map(lambda v: S(v).upper() == "TRUE")
        upw_known = out["upw_is_oa"].notna()
        out["oa_disagree"] = upw_known & (out["upw_is_oa"].astype("boolean") != oa_guess)
        out["oa_final"] = out["upw_is_oa"].astype("boolean").fillna(oa_guess)
    else:
        out["oa_final"] = out["upw_is_oa"]

    out.to_parquet(OUT_PARQUET, index=False)
    print(f"[write] {OUT_PARQUET}", file=sys.stderr)

    upw_cov = out["upw_is_oa"].notna().mean()
    oc_cov = out["oc_cited_by"].notna().mean()
    print(f"[coverage] Unpaywall OA resolved: {upw_cov:.1%}; "
          f"OpenCitations count: {oc_cov:.1%}", file=sys.stderr)
    if "oa_disagree" in out.columns:
        print(f"[oa] OpenAlex vs Unpaywall disagree on {out['oa_disagree'].mean():.1%} "
              f"of DOI-resolved rows", file=sys.stderr)
    return out


# =========================================================================== #
if __name__ == "__main__":
    df_all = pd.read_parquet(IN_PARQUET)
    df = df_all[df_all["llm_label"].notna()]
    print(f"[in] {len(df):,} rows", file=sys.stderr)
    upw = harvest_unpaywall(df)
    oc  = harvest_opencitations(df)
    reconcile(df, upw, oc)

In [ ]:
upw

In [ ]:
oc

In [ ]:
merge = reconcile(df, upw, oc)
merge